#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [54]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader=TextLoader("speech.txt")
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=30)
docs=text_splitter.split_documents(documents)


In [55]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\nâ€¦'),
 Document(metadata={'source': 'speech.txt'}, page_content='â€¦\n\nIt will be all the easier for us to conduct

In [56]:
embeddings=OllamaEmbeddings(model="nomic-embed-text")
db=FAISS.from_documents(docs,embeddings)
db

In [57]:
### querying 
query="How does the speaker describe the desired outcome of the war?"
docs=db.similarity_search(query)
# docs
docs[0].page_content


'â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [58]:
retriever=db.as_retriever()
docs=retriever.invoke(query)
docs[0].page_content

'â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [59]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='f489b0e5-ffae-411d-b3ff-6efeaac00738', metadata={'source': 'speech.txt'}, page_content='â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
  np.float32(0.86928004)),
 (Document(id='9c1011a5-6d77-418c-b788-bb1ba7d16d1e', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have perfo

In [60]:
embedding_vector=embeddings.embed_query(query)
embedding_vector

[-0.018022666,
 0.0063378103,
 -0.15976362,
 -0.04520547,
 0.1010064,
 0.072802484,
 -0.04555546,
 0.006414784,
 0.01743762,
 -0.0009013847,
 -0.008588354,
 0.029710604,
 0.044366676,
 0.06834126,
 0.09486338,
 -0.044027608,
 0.014219053,
 -0.045692984,
 -0.035663854,
 0.021440174,
 -0.029523956,
 -0.012377359,
 3.4388548e-05,
 0.031195113,
 0.0716284,
 0.02461166,
 -0.020600228,
 0.039740924,
 -0.03854113,
 -0.036187973,
 0.04562628,
 0.004831273,
 -0.006412556,
 0.0043895436,
 -0.072305374,
 -0.07704111,
 0.0037011516,
 0.062248457,
 0.013758106,
 -0.05989637,
 -0.006053241,
 -0.026577953,
 0.0018722832,
 -0.07702947,
 0.055040732,
 -0.020058362,
 -0.018467348,
 0.005958723,
 -6.0951323e-05,
 -0.027846962,
 -0.035668477,
 -0.029609242,
 0.0048507503,
 -0.010885337,
 0.057130765,
 0.0062765433,
 -0.03355526,
 -0.034634635,
 0.031632584,
 0.0029258549,
 0.024623457,
 0.01023654,
 -0.026977414,
 0.037538726,
 0.059696984,
 -0.0315908,
 -0.060434762,
 0.029940791,
 0.025176018,
 -0.05914

In [61]:
docs_score=db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='f489b0e5-ffae-411d-b3ff-6efeaac00738', metadata={'source': 'speech.txt'}, page_content='â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='9c1011a5-6d77-418c-b788-bb1ba7d16d1e', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. 

In [62]:
### Saving And Loading
db.save_local("faiss_index")

In [63]:
new_db=FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True)
docs=new_db.similarity_search(query)

In [64]:
docs

[Document(id='f489b0e5-ffae-411d-b3ff-6efeaac00738', metadata={'source': 'speech.txt'}, page_content='â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='9c1011a5-6d77-418c-b788-bb1ba7d16d1e', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. 